In [3]:
import numpy as np
from scipy.optimize import minimize, differential_evolution
import warnings
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — DATA TARGET J PER MOLEKUL (INPUT)
# ══════════════════════════════════════════════════════════════════════════════


targets_raw = {
    "23dimethylbutane"   :   (0.140034696, "co2"),
    "24dimethylpentane"  :   (0.062216133, "co2"),
    "2methylpropane"     :   (0.121309801, "co2"),
    "etanol"             :   (0.044110593, "co2"),
    "propanol"           :   (0.014219607, "co2"),
    "butanol"            :   (0.040411697, "co2"),
    "acetic_acid"        :   (-0.003886209, "co2"),
    "propanoic_acid"     :   (0.075011232, "co2"),
    "butanoic_acid"      :   (0.096987792, "co2"),
    "methylamine"        :   (-0.26485945, "water"),
    "ethylamine"         :   (-0.130009497, "water"),
    "propylamine"        :   (-0.114436399, "water"),
    
}

molecules = [
    "23dimethylbutane",
    "24dimethylpentane",
    "2methylpropane",
    "etanol",
    "propanol",
    "butanol",
    "acetic_acid",
    "propanoic_acid",
    "butanoic_acid",
    "methylamine",
    "ethylamine",
    "propylamine",
]

targets = np.array([targets_raw[m][0] for m in molecules], dtype=float)
solvent_mapping = [targets_raw[m][1] for m in molecules]


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — NILAI J_GRUP YANG SUDAH DIKETAHUI (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

J_known = {
    "CH3" : 12.992,   
    "CH2" : 7.345,    
}

# Nilai Pseudo-Ionization Energy dari pelarut
J_solvent = {
    "water" : 9.7,
    "co2"   : 13.78,
}


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — GRUP YANG DI-FITTING + BOUNDS (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

J_unknown = {
    "CH"   : (0.000001,  1000.0),
    "OH"   : (0.000001,  1000.0),
    "COOH" : (0.000001,  1000.0),
    "NH2"  : (0.000001,  1000.0),

}

unknown_names  = list(J_unknown.keys())
bounds_unknown = list(J_unknown.values())
N_unk = len(unknown_names)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — JUMLAH GRUP PER MOLEKUL (INPUT)
# ══════════════════════════════════════════════════════════════════════════════

n_groups = np.array([
    # CH3   CH2   CH    OH    COOH  NH2
    # 23dimethylbutane  
    [  4,    0,    2,    0,    0,    0  ],
    # 24dimethylpentane 
    [  4,    1,    2,    0,    0,    0  ],
    # 2methylpropane    
    [  3,    0,    1,    0,    0,    0  ],
    # etanol            
    [  1,    1,    0,    1,    0,    0  ],
    # propanol          
    [  1,    2,    0,    1,    0,    0  ],
    # butanol          
    [  1,    3,    0,    1,    0,    0  ],
    # acetic_acid       
    [  1,    0,    0,    0,    1,    0  ],
    # propanoic_acid    
    [  1,    1,    0,    0,    1,    0  ],
    # butanoic_acid     
    [  1,    2,    0,    0,    1,    0  ],
    # methylamine       
    [  1,    0,    0,    0,    0,    1  ],
    # ethylamine        
    [  1,    1,    0,    0,    0,    1  ],
    # propylamine       
    [  1,    2,    0,    0,    0,    1  ],

], dtype=float)


all_group_names = list(J_known.keys()) + unknown_names


assert n_groups.shape == (len(molecules), len(all_group_names)), (
    f"Dimensi n_groups {n_groups.shape} harus = "
    f"({len(molecules)} molekul × {len(all_group_names)} grup). "
    f"Periksa jumlah baris (= jumlah molekul) dan "
    f"jumlah kolom (= jumlah known + unknown groups)."
)


n_total = n_groups.sum(axis=1)


n_known_groups = len(J_known)
idx_known   = list(range(n_known_groups))
idx_unknown = list(range(n_known_groups, len(all_group_names)))


J_known_vals = np.array(list(J_known.values()))


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — MODEL (OTOMATIS — TIDAK PERLU DIEDIT)
# ══════════════════════════════════════════════════════════════════════════════

def model(mol_idx, x):

    J_all = np.concatenate([J_known_vals, x])
    n_k = n_groups[mol_idx]


    product = np.prod(J_all ** n_k)
    J_j = product ** (1.0 / n_total[mol_idx])
    

    nama_pelarut = solvent_mapping[mol_idx]
    J_i = J_solvent[nama_pelarut]


    sisi_kanan = (2.0 * np.sqrt(J_i * J_j)) / (J_i + J_j)
    

    k_ij_tgt = targets[mol_idx]
    if k_ij_tgt < 0:
        k_ij_prediksi = 1.0 - sisi_kanan
    else:
        k_ij_prediksi = 1.0 - sisi_kanan

    return k_ij_prediksi



# ══════════════════════════════════════════════════════════════════════════════
# objective function
# ══════════════════════════════════════════════════════════════════════════════

def objective(x):
    F = 0.0
    for mol_idx in range(len(molecules)):
        k_ij_GC  = model(mol_idx, x)
        k_ij_tgt = targets[mol_idx]
        if abs(k_ij_tgt) > 1e-15:
            F += ((k_ij_tgt - k_ij_GC) / k_ij_tgt) ** 2
        else:
            F += (k_ij_tgt - k_ij_GC) ** 2
    return F



# ── Optimasi ─────────────────────────────────────────────────────────────────
print("=" * 62)
print("  PC-SAFT kij — Group Contribution Fitting")
print("=" * 62)
print(f"\n  Molekul  : {len(molecules)}")
print(f"  Known    : {', '.join(f'{k}={v}' for k, v in J_known.items())}")
print(f"  Unknown  : {', '.join(unknown_names)}")
print(f"  Total    : {N_unk} parameter\n")

print("  Stage 1: Global search (differential_evolution)...")
res_g = differential_evolution(
    objective, bounds_unknown,
    seed=42, maxiter=8000, tol=1e-14,
    popsize=25, mutation=(0.5, 1.5), recombination=0.9,
    polish=True, disp=False,
)

print("  Stage 2: Local refinement (L-BFGS-B)...")
res_l = minimize(
    objective, res_g.x,
    method="L-BFGS-B", bounds=bounds_unknown,
    options={"maxiter": 100000, "ftol": 1e-16, "gtol": 1e-13},
)

x_opt = res_l.x if res_l.fun < res_g.fun else res_g.x
F_opt = min(res_l.fun, res_g.fun)


# ── Cetak hasil ───────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  FITTED PARAMETERS")
print("=" * 62)
print("\n  Known (tidak di-fitting):")
for name, val in J_known.items():
    print(f"    J_{name:<8} = {val:>12.6f}  [tetap]")
print("\n  Unknown (hasil fitting):")
for j, name in enumerate(unknown_names):
    print(f"    J_{name:<8} = {x_opt[j]:>12.6f}")

print(f"\n  F_OBJ (final) = {F_opt:.6e}")


# ── Validasi ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  VALIDATION — Target vs. GC-Predicted (APD%)")
print("=" * 62)
print(f"  {'Molekul':<22} {'kij_target':>12} {'kij_GC':>12} {'APD%':>9}")
print(f"  {'-'*58}")

apd_list = []
for mol_idx, mol in enumerate(molecules):
    J_tgt = targets[mol_idx]
    J_gc  = model(mol_idx, x_opt)
    apd   = 100.0 * abs(J_tgt - J_gc) / abs(J_tgt)
    apd_list.append(apd)
    print(f"  {mol:<22} {J_tgt:>12.6f} {J_gc:>12.6f} {apd:>8.4f}%")

print(f"\n  Mean APD% = {np.mean(apd_list):.4f}%")


# ── Tabel ringkasan ───────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  SUMMARY TABLE (copy-paste ready)")
print("=" * 62)
print(f"  {'Grup':<10} {'J_grup':>14}  {'Status'}")
print("  " + "-" * 36)
for name, val in J_known.items():
    print(f"  {name:<10} {val:>14.6f}  known")
for j, name in enumerate(unknown_names):
    print(f"  {name:<10} {x_opt[j]:>14.6f}  fitted")


# ── Diagnostik batas ─────────────────────────────────────────────────────────
print("\n" + "=" * 62)
print("  DIAGNOSTIC — Bound Check")
print("=" * 62)

bound_hit = False
for i, (val, (lo, hi)) in enumerate(zip(x_opt, bounds_unknown)):
    tol = max(1e-4, 1e-4 * abs(hi - lo))
    if abs(val - lo) < tol or abs(val - hi) < tol:
        print(f"  ⚠  J_{unknown_names[i]} = {val:.6f} menempel di batas [{lo}, {hi}].")
        print(f"      → Perlebar batas ini di SECTION 3 dan jalankan ulang.")
        bound_hit = True
if not bound_hit:
    print("  ✓  Tidak ada parameter yang menempel di batas. Solusi interior.")

  PC-SAFT kij — Group Contribution Fitting

  Molekul  : 12
  Known    : CH3=12.992, CH2=7.345
  Unknown  : CH, OH, COOH, NH2
  Total    : 4 parameter

  Stage 1: Global search (differential_evolution)...
  Stage 2: Local refinement (L-BFGS-B)...

  FITTED PARAMETERS

  Known (tidak di-fitting):
    J_CH3      =    12.992000  [tetap]
    J_CH2      =     7.345000  [tetap]

  Unknown (hasil fitting):
    J_CH       =   367.599181
    J_OH       =   196.997619
    J_COOH     =    14.261438
    J_NH2      =     9.124534

  F_OBJ (final) = 6.794295e+00

  VALIDATION — Target vs. GC-Predicted (APD%)
  Molekul                  kij_target       kij_GC      APD%
  ----------------------------------------------------------
  23dimethylbutane           0.140035     0.124709  10.9444%
  24dimethylpentane          0.062216     0.077591  24.7124%
  2methylpropane             0.121310     0.070957  41.5077%
  etanol                     0.044111     0.051677  17.1526%
  propanol                   0.0